In [2]:
# 1. Install libraries
import pandas as pd
import numpy as np
import sqlite3
import matplotlib.pyplot as plt

from pathlib import Path


In [3]:
DATA_PATH = Path("data/raw.csv")

df = pd.read_csv(DATA_PATH)

print(df.shape)
df.head()

(73100, 15)


,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Demand Forecast,Price,Discount,Weather Condition,Holiday/Promotion,Competitor Pricing,Seasonality
0,2022-01-01,S001,P0001,Groceries,North,231,127,55,135.47,33.50,20,Rainy,0,29.69,Autumn
1,2022-01-01,S001,P0002,Toys,South,204,150,66,144.04,63.01,20,Sunny,0,66.16,Autumn
2,2022-01-01,S001,P0003,Toys,West,102,65,51,74.02,27.99,10,Sunny,1,31.32,Summer
3,2022-01-01,S001,P0004,Toys,North,469,61,164,62.18,32.72,10,Cloudy,1,34.74,Autumn
4,2022-01-01,S001,P0005,Electronics,East,166,14,135,9.26,73.64,0,Sunny,0,68.95,Summer


In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 73100 entries, 0 to 73099
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Date                73100 non-null  str    
 1   Store ID            73100 non-null  str    
 2   Product ID          73100 non-null  str    
 3   Category            73100 non-null  str    
 4   Region              73100 non-null  str    
 5   Inventory Level     73100 non-null  int64  
 6   Units Sold          73100 non-null  int64  
 7   Units Ordered       73100 non-null  int64  
 8   Demand Forecast     73100 non-null  float64
 9   Price               73100 non-null  float64
 10  Discount            73100 non-null  int64  
 11  Weather Condition   73100 non-null  str    
 12  Holiday/Promotion   73100 non-null  int64  
 13  Competitor Pricing  73100 non-null  float64
 14  Seasonality         73100 non-null  str    
dtypes: float64(3), int64(5), str(7)
memory usage: 8.4 MB


In [10]:
df.columns

Index(['Date', 'Store ID', 'Product ID', 'Category', 'Region',
       'Inventory Level', 'Units Sold', 'Units Ordered', 'Demand Forecast',
       'Price', 'Discount', 'Weather Condition', 'Holiday/Promotion',
       'Competitor Pricing', 'Seasonality'],
      dtype='str')

In [11]:
df.describe(include="all")

,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Demand Forecast,Price,Discount,Weather Condition,Holiday/Promotion,Competitor Pricing,Seasonality
count,73100,73100,73100,73100,73100,73100.000000,73100.000000,73100.000000,73100.000000,73100.000000,73100.000000,73100,73100.000000,73100.000000,73100
unique,731,5,20,5,4,NaN,NaN,NaN,NaN,NaN,NaN,4,NaN,NaN,4
top,2022-01-01,S001,P0001,Furniture,East,NaN,NaN,NaN,NaN,NaN,NaN,Sunny,NaN,NaN,Spring
freq,100,14620,3655,14699,18349,NaN,NaN,NaN,NaN,NaN,NaN,18290,NaN,NaN,18317
mean,NaN,NaN,NaN,NaN,NaN,274.469877,136.464870,110.004473,141.494720,55.135108,10.009508,NaN,0.497305,55.146077,NaN
std,NaN,NaN,NaN,NaN,NaN,129.949514,108.919406,52.277448,109.254076,26.021945,7.083746,NaN,0.499996,26.191408,NaN
min,NaN,NaN,NaN,NaN,NaN,50.000000,0.000000,20.000000,-9.990000,10.000000,0.000000,NaN,0.000000,5.030000,NaN
25%,NaN,NaN,NaN,NaN,NaN,162.000000,49.000000,65.000000,53.670000,32.650000,5.000000,NaN,0.000000,32.680000,NaN
50%,NaN,NaN,NaN,NaN,NaN,273.000000,107.000000,110.000000,113.015000,55.050000,10.000000,NaN,0.000000,55.010000,NaN
75%,NaN,NaN,NaN,NaN,NaN,387.000000,203.000000,155.000000,208.052500,77.860000,15.000000,NaN,1.000000,77.820000,NaN


## Data Preprocessing

In [12]:
df = df.copy()

# convert column names to snake_case
df.columns = (df.columns
              .str.strip()
              .str.lower()
              .str.replace(" ", "_")
              .str.replace("/", "_")
              )
df.columns

Index(['date', 'store_id', 'product_id', 'category', 'region',
       'inventory_level', 'units_sold', 'units_ordered', 'demand_forecast',
       'price', 'discount', 'weather_condition', 'holiday_promotion',
       'competitor_pricing', 'seasonality'],
      dtype='str')

In [13]:
df["date"] = pd.to_datetime(df["date"])

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 73100 entries, 0 to 73099
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   date                73100 non-null  datetime64[us]
 1   store_id            73100 non-null  str           
 2   product_id          73100 non-null  str           
 3   category            73100 non-null  str           
 4   region              73100 non-null  str           
 5   inventory_level     73100 non-null  int64         
 6   units_sold          73100 non-null  int64         
 7   units_ordered       73100 non-null  int64         
 8   demand_forecast     73100 non-null  float64       
 9   price               73100 non-null  float64       
 10  discount            73100 non-null  int64         
 11  weather_condition   73100 non-null  str           
 12  holiday_promotion   73100 non-null  int64         
 13  competitor_pricing  73100 non-null  float64       
 14  s

### EDA 

In [14]:
df["category"].value_counts()

category
Furniture      14699
Toys           14643
Clothing       14626
Groceries      14611
Electronics    14521
Name: count, dtype: int64

In [15]:
df["region"].value_counts()

region
East     18349
South    18297
North    18228
West     18226
Name: count, dtype: int64

In [16]:
df["holiday_promotion"].value_counts()

holiday_promotion
0    36747
1    36353
Name: count, dtype: int64

In [17]:
df["seasonality"].value_counts()

seasonality
Spring    18317
Summer    18305
Winter    18285
Autumn    18193
Name: count, dtype: int64

In [18]:
df["category"] = df["category"].replace({
    "Groceries": "Beverages"
})

In [19]:
df["category"].value_counts()

category
Furniture      14699
Toys           14643
Clothing       14626
Beverages      14611
Electronics    14521
Name: count, dtype: int64

In [20]:
df.isnull().sum()

date                  0
store_id              0
product_id            0
category              0
region                0
inventory_level       0
units_sold            0
units_ordered         0
demand_forecast       0
price                 0
discount              0
weather_condition     0
holiday_promotion     0
competitor_pricing    0
seasonality           0
dtype: int64

In [21]:
df.to_csv("data/clean_inventory.csv", index=False)

print("Dataset cleaned and saved successfully.")

Dataset cleaned and saved successfully.


## The DataBase

In [22]:
import sqlite3

In [ ]:
DATABASE = "business_analytics.db"

conn = sqlite3.connect(DATABASE)

df.to_sql(
    "sales_inventory",
    conn,
    if_exists="replace",
    index=False
)

print("Database created successfully!")

Database created successfully!


In [56]:
query = """
WITH promo_bev AS (
    SELECT
        region,
        SUM(CAST(units_sold AS INTEGER)) AS total_sold
    FROM sales_inventory
    WHERE category = 'Beverages'
      AND holiday_promotion = 1
    GROUP BY region
)
SELECT region
FROM promo_bev
ORDER BY total_sold DESC
LIMIT 1;
"""

pd.read_sql(query, conn)

,region
0,East


In [35]:
query = """
SELECT
category,
AVG(inventory_level) AS avg_inventory
FROM sales_inventory
GROUP BY category
ORDER BY avg_inventory DESC;
"""

pd.read_sql(query, conn)

,category,avg_inventory
0,Furniture,275.816246
1,Beverages,275.755595
2,Clothing,274.597771
3,Toys,273.646862
4,Electronics,272.514427


Until now we are able to get the SQL queries from the dataset 
### Phase -1 Completed

# Phase -2

In [37]:
# Loading the API key
import os
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("OPENROUTER_API_KEY")

print(API_KEY[:15] + "...")


sk-or-v1-5241de...


In [38]:
from openai import OpenAI

client = OpenAI(
    api_key=API_KEY,
    base_url="https://openrouter.ai/api/v1"
)

In [45]:
MODEL = "nvidia/nemotron-3-ultra-550b-a55b:free"

In [46]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": "Say hello in one sentence."
        }
    ]
)

print(response.choices[0].message.content)

Hello there!


In [47]:
from openai import OpenAI

print(OpenAI)

<class 'openai.OpenAI'>


In [48]:
import openai
print(openai.__version__)

2.46.0


`Schema`

In [ ]:
# SCHEMA = """
# Database Name:
# business_analytics.db

# Table:
# sales_inventory

# Columns:

# date (DATE)
# store_id (TEXT)
# product_id (TEXT)
# category (TEXT)
# region (TEXT)
# inventory_level (INTEGER)
# units_sold (INTEGER)
# units_ordered (INTEGER)
# demand_forecast (REAL)
# price (REAL)
# discount (INTEGER)
# weather_condition (TEXT)
# holiday_promotion (INTEGER)
# competitor_pricing (REAL)
# seasonality (TEXT)

# Business Glossary

# Promotion or Campaign
# → holiday_promotion = 1

# No Promotion
# → holiday_promotion = 0

# Sales
# → SUM(units_sold)

# Revenue
# → SUM(units_sold * price)

# Average Inventory
# → AVG(inventory_level)

# Inventory Reduction
# → Compare average inventory during promotions
# against non-promotion periods.

# Region
# → North, South, East, West

# Categories
# → Beverages
# → Electronics
# → Clothing
# → Furniture
# → Toys
# """

In [ ]:
# SYSTEM_PROMPT = f"""
# You are an expert Business Intelligence SQL Assistant.

# Your job is to convert business questions into SQLite SQL queries.

# Database Schema:

# {SCHEMA}

# Rules:

# 1. Generate ONLY SQLite SQL.

# 2. Only use the table sales_inventory.

# 3. Never use tables that do not exist.

# 4. Never generate INSERT, UPDATE, DELETE, DROP or ALTER statements.

# 5. Return ONLY the SQL query.

# 6. Do not explain anything.

# 7. Always use valid SQLite syntax.
# """

In [ ]:
# def generate_sql(question):

#     response = client.chat.completions.create(
#         model=MODEL,
#         temperature=0,
#         messages=[
#             {
#                 "role": "system",
#                 "content": SYSTEM_PROMPT
#             },
#             {
#                 "role": "user",
#                 "content": question
#             }
#         ]
#     )

#     return response.choices[0].message.content

In [72]:
question = "Which region sold the most beverages during promotions?"

sql = generate_sql(question)

print(sql)

SELECT 
    region,
    SUM(CAST(units_sold AS INTEGER)) AS total_sold
FROM sales_inventory
WHERE category = 'Beverages'
  AND holiday_promotion = 1
GROUP BY region
ORDER BY total_sold DESC
LIMIT 1;


*Now we have the LLM that converts Natural Language -> SQL Query*
* But we cannot Blindly Trust AI 
* In enterprise AI we NEVER execute AI-generated SQL

### SQL Validator

In [ ]:
# import re

# def clean_sql(sql):

#     sql = sql.replace("```sql", "")
#     sql = sql.replace("```", "")
#     sql = sql.strip()

#     return sql

In [ ]:
# def validate_sql(sql):

#     sql_lower = sql.lower()

#     blocked = [
#         "insert",
#         "update",
#         "delete",
#         "drop",
#         "alter",
#         "truncate",
#         "create",
#         "attach",
#         "pragma"
#     ]

#     if not sql_lower.startswith(("select", "with")):
#         return False, "Only SELECT or WITH queries are allowed."

#     for word in blocked:
#         if word in sql_lower:
#             return False, f"Blocked keyword detected: {word}"

#     if "sales_inventory" not in sql_lower:
#         return False, "Query must reference the sales_inventory table."

#     return True, "SQL is valid."

In [61]:
sql = clean_sql(sql)

is_valid, message = validate_sql(sql)

print(is_valid)
print(message)

True
SQL is valid.


In [ ]:
# # this executes the validated SQL query
# def execute_sql(sql):

#     return pd.read_sql_query(sql, conn)

In [63]:
execute_sql(sql)

,region
0,East


### Phase -2 Completed

In [65]:
# test =
question = "what was the weekly sales trend for beverages last quarter?"
sql = generate_sql(question)
cleaned_sql = clean_sql(sql)
if validate_sql(cleaned_sql):
    print(execute_sql(cleaned_sql))
else:
    print(validate_sql(cleaned_sql))

       week  weekly_sales
0   2023-39          2496
1   2023-40         22297
2   2023-41         20058
3   2023-42         19495
4   2023-43         17783
5   2023-44         22733
6   2023-45         16855
7   2023-46         15630
8   2023-47         16676
9   2023-48         15279
10  2023-49         19172
11  2023-50         20430
12  2023-51         16189
13  2023-52         18158
14  2024-01          2227


Natural Language _to_ SQL-Query to _to_ Output ✅
## Phase -3 Completed

Building a good response(output) for user 

In [ ]:
# def generate_business_insight(question, result_df):

#     table = result_df.to_string(index=False)

#     prompt = f"""
#             You are a Business Intelligence Analyst.
#             A business user asked:

#             {question}

#             The SQL query has already been executed.

#             The result is:

#             {table}

#             Instructions:

#             1. Use ONLY the numbers shown.
#             2. Never invent values.
#             3. Give a concise business explanation.
#             4. Mention trends if visible.
#             5. Keep the response under 120 words.
#         """

#     response = client.chat.completions.create(
#         model=MODEL,
#         temperature=0,
#         messages=[
#             {
#                 "role": "user",
#                 "content": prompt
#             }
#         ]
#     )

#     return response.choices[0].message.content

`one function for entire architecture`

In [79]:
def extract_metrics(df):

    metrics = {}

    metrics["Rows Returned"] = len(df)

    metrics["Columns"] = len(df.columns)

    return metrics

In [82]:
from utils.prompts import SCHEMA, SYSTEM_PROMPT

from utils.database import (
    clean_sql,
    validate_sql,
    execute_sql
)

from utils.ai import (
    generate_sql,
    generate_business_insight
)

def ask_business_assistant(question):

    print("="*80)
    print("QUESTION")
    print(question)

    print("\nGenerating SQL...")

    sql = generate_sql(question)

    sql = clean_sql(sql)

    valid, message = validate_sql(sql)

    if not valid:

        print(message)

        return

    print("\nGenerated SQL")
    print(sql)

    print("\nExecuting Query...")

    result = execute_sql(sql)

    print("\nQuery Result")

    display(result)

    print("\nGenerating Business Insight...")

    insight = generate_business_insight(question, result)

    metric = extract_metrics(result)
    print(metric)
    print("\nBusiness Insight\n")

    print(insight)

    return {
        "question": question,
        "sql": sql,
        "table": result,
        "metric": metric,
        "insight": insight
    }

In [ ]:
answer = ask_business_assistant("Which region sold the most beverages during promotions?")

QUESTION
Which region sold the most beverages during promotions?

Generating SQL...


TypeError: 'NoneType' object is not subscriptable

In [84]:
answer = ask_business_assistant("what  region has sold many beverages while promotions?")

QUESTION
what  region has sold many beverages while promotions?

Generating SQL...

Generated SQL
SELECT region, SUM(units_sold) AS total_sold
FROM sales_inventory
WHERE category = 'Beverages' AND holiday_promotion = 1
GROUP BY region
ORDER BY total_sold DESC
LIMIT 1;

Executing Query...

Query Result


,region,total_sold
0,East,258484



Generating Business Insight...
{'Rows Returned': 1, 'Columns': 2}

Business Insight

The **East region** leads with **258,484 beverages sold during promotions**. Since the result set contains only this single row, it indicates East is the top-performing region in the current filtered view (likely the highest volume or the only region meeting the promotion criteria). No comparative trend across other regions can be derived from this output alone.


## Phase-4 Completed🎉

### Phase-5 Product Engineering 😎

In [87]:
from utils.planner import plan_analysis

plan = plan_analysis("What is the total revenue last month?")

print(plan)

{'intent': 'single_value', 'layout': [{'type': 'metric_cards'}, {'type': 'summary'}]}


In [1]:
import os
from huggingface_hub import InferenceClient

client = InferenceClient(
    api_key=os.environ["HF_TOKEN"],
)

completion = client.chat.completions.create(
    model="MiniMaxAI/MiniMax-M3:novita",
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": "Describe this image in one sentence."
                },
                {
                    "type": "image_url",
                    "image_url": {
                        "url": "https://cdn.britannica.com/61/93061-050-99147DCE/Statue-of-Liberty-Island-New-York-Bay.jpg"
                    }
                }
            ]
        }
    ],
)

print(completion.choices[0].message)

c:\Users\sunain\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ChatCompletionOutputMessage(role='assistant', content='The Statue of Liberty stands proudly on Liberty Island in the foreground, with the iconic Manhattan skyline—including the Empire State Building—stretching across the background under a clear sky bathed in warm golden light.', reasoning=None, tool_call_id=None, tool_calls=None, reasoning_content='The user wants a one-sentence description of the image. Let me analyze the image:\n\nThe image shows the Statue of Liberty standing on its pedestal on Liberty Island, with the Manhattan skyline in the background, including the Empire State Building. The scene is bathed in warm golden light, likely during sunset or early morning, with calm water in the foreground.', reasoning_details=[{'type': 'reasoning.text', 'text': 'The user wants a one-sentence description of the image. Let me analyze the image:\n\nThe image shows the Statue of Liberty standing on its pedestal on Liberty Island, with the Manhattan skyline in the background, including th

In [1]:
!pip install huggingface_hub



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
from utils.database import execute_sql
from utils.ai import generate_sql
question = "Which categories saw inventory reduction during promotions?"

query = generate_sql(question)

# SELECT SUM(units_sold) AS total_sales
# FROM sales_inventory
# WHERE region = 'South'
#   AND strftime('%Y-%m', date) = strftime('%Y-%m', date('now', '-1 month'));
# '''

# execute_sql(query)

In [8]:
q = """SELECT 
    category,
    AVG(CASE WHEN holiday_promotion = 1 THEN inventory_level END) AS avg_inventory_promotion,
    AVG(CASE WHEN holiday_promotion = 0 THEN inventory_level END) AS avg_inventory_no_promotion
FROM sales_inventory
GROUP BY category
HAVING AVG(CASE WHEN holiday_promotion = 1 THEN inventory_level END) 
       < AVG(CASE WHEN holiday_promotion = 0 THEN inventory_level END);"""

execute_sql(q)

,category,avg_inventory_promotion,avg_inventory_no_promotion
0,Electronics,272.103396,272.920339
1,Furniture,275.571390,276.061870
2,Toys,273.619451,273.673436
